In [1]:
!pip install -U lightning torchmetrics torchvision

import os
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import transforms

import lightning.pytorch as pl
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor

from torchmetrics.classification import MulticlassF1Score, MulticlassAUROC

SEED = 42
pl.seed_everything(SEED, workers=True)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 851.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


In [2]:
@dataclass
class DataConfig:
    data_dir: str = "./data"
    batch_size: int = 128
    num_workers: int = 2
    val_split: float = 0.1
    mean: float = 0.2860
    std: float = 0.3530


class FashionMNISTDataModule(pl.LightningDataModule):
    def __init__(self, cfg: DataConfig, seed: int = 42):
        super().__init__()
        self.cfg = cfg
        self.seed = seed

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((self.cfg.mean,), (self.cfg.std,))
        ])

        self.train_ds = None
        self.val_ds = None
        self.test_ds = None

    def prepare_data(self):
        torchvision.datasets.FashionMNIST(self.cfg.data_dir, train=True, download=True)
        torchvision.datasets.FashionMNIST(self.cfg.data_dir, train=False, download=True)

    def setup(self, stage: str | None = None):
        full_train = torchvision.datasets.FashionMNIST(
            self.cfg.data_dir, train=True, transform=self.transform, download=False
        )
        self.test_ds = torchvision.datasets.FashionMNIST(
            self.cfg.data_dir, train=False, transform=self.transform, download=False
        )

        val_size = int(len(full_train) * self.cfg.val_split)
        train_size = len(full_train) - val_size

        g = torch.Generator().manual_seed(self.seed)
        self.train_ds, self.val_ds = random_split(full_train, [train_size, val_size], generator=g)

    def _dl_kwargs(self):
        pin = torch.cuda.is_available()
        return dict(
            batch_size=self.cfg.batch_size,
            num_workers=self.cfg.num_workers,
            pin_memory=pin,
            persistent_workers=(self.cfg.num_workers > 0),
        )

    def train_dataloader(self):
        return DataLoader(self.train_ds, shuffle=True, **self._dl_kwargs())

    def val_dataloader(self):
        return DataLoader(self.val_ds, shuffle=False, **self._dl_kwargs())

    def test_dataloader(self):
        return DataLoader(self.test_ds, shuffle=False, **self._dl_kwargs())


cfg = DataConfig(
    data_dir="./data",
    batch_size=128,
    num_workers=min(4, os.cpu_count() or 2),
    val_split=0.1,
)

dm = FashionMNISTDataModule(cfg, seed=SEED)


In [3]:
class FashionMNISTModel(pl.LightningModule):
    def __init__(self, lr: float = 1e-3, weight_decay: float = 1e-4, dropout_p: float = 0.3):
        super().__init__()
        self.save_hyperparameters()

        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, 10),
        )

        self.criterion = nn.CrossEntropyLoss()

        self.val_f1 = MulticlassF1Score(num_classes=10, average="macro")
        self.val_auc = MulticlassAUROC(num_classes=10, average="macro")

        self.test_f1 = MulticlassF1Score(num_classes=10, average="macro")
        self.test_auc = MulticlassAUROC(num_classes=10, average="macro")

    def forward(self, x):
        return self.net(x)

    def _shared_step(self, batch):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        probs = torch.softmax(logits, dim=1)
        return loss, probs, y

    def training_step(self, batch, batch_idx):
        loss, probs, y = self._shared_step(batch)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, probs, y = self._shared_step(batch)
        self.val_f1.update(probs, y)
        self.val_auc.update(probs, y)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

    def on_validation_epoch_end(self):
        f1 = self.val_f1.compute()
        auc = self.val_auc.compute()
        self.log("val_f1", f1, prog_bar=True)
        self.log("val_auc", auc, prog_bar=True)
        self.val_f1.reset()
        self.val_auc.reset()

    def test_step(self, batch, batch_idx):
        loss, probs, y = self._shared_step(batch)
        self.test_f1.update(probs, y)
        self.test_auc.update(probs, y)
        self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True)

    def on_test_epoch_end(self):
        f1 = self.test_f1.compute()
        auc = self.test_auc.compute()
        self.log("test_f1", f1, prog_bar=True)
        self.log("test_auc", auc, prog_bar=True)
        self.test_f1.reset()
        self.test_auc.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1,
            },
        }


model = FashionMNISTModel(lr=1e-3, weight_decay=1e-4, dropout_p=0.3)


In [4]:
logger = TensorBoardLogger(save_dir="tb_logs", name="fashionmnist")

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    mode="min",
    min_delta=1e-3,
)

ckpt = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="best-{epoch:02d}-{val_loss:.4f}",
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices="auto",
    logger=logger,
    callbacks=[early_stop, ckpt, lr_monitor],
    log_every_n_steps=50,
    deterministic=True,
)

trainer.fit(model, datamodule=dm)

print("Best checkpoint:", ckpt.best_model_path)


INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
100%|██████████| 26.4M/26.4M [00:02<00:00, 13.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 211kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.92MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.22MB/s]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net       │ Sequential        │  421 K │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss  │      0 │ train │     0 │
│ 2 │ val_f1    │ MulticlassF1Score │      0 │ train │     0 │
│ 3 │ val_auc   │ MulticlassAUROC   │      0 │ train │     0 │
│ 4 │ test_f1   │ MulticlassF1Score │      0 │ train │     0 │
│ 5 │ test_auc  │ MulticlassAUROC   │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 421 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 421 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 19                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


Best checkpoint: tb_logs/fashionmnist/version_0/checkpoints/best-epoch=09-val_loss=0.2060.ckpt


In [5]:
best_model = FashionMNISTModel.load_from_checkpoint(ckpt.best_model_path)

test_metrics = trainer.test(best_model, datamodule=dm, verbose=True)
test_metrics


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_auc          │    0.9956879615783691     │
│          test_f1          │    0.9236930012702942     │
│         test_loss         │    0.2199811488389969     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.2199811488389969,
  'test_f1': 0.9236930012702942,
  'test_auc': 0.9956879615783691}]